1.เตรียมพารามิเตอร์ Charuco

In [25]:
import cv2
import numpy as np
import glob
import os

aruco = cv2.aruco

# ====== Charuco board params (ต้องตรงกับที่ใช้ generate A3) ======
SQUARES_X = 10          # cols
SQUARES_Y = 7           # rows
SQUARE_LEN = 30.0       # mm (scale ไม่สำคัญ แค่สัดส่วนถูก)
MARKER_LEN = 18.0       # mm

dictionary = aruco.getPredefinedDictionary(aruco.DICT_6X6_250)
board = aruco.CharucoBoard(
    (SQUARES_X, SQUARES_Y),
    SQUARE_LEN,
    MARKER_LEN,
    dictionary
)


2.อ่านรูปจากโฟลเดอร์ + หา Charuco corners

In [26]:
# ----- แก้ path ตามของคุณ -----
LEFT_DIR  = r"D:\FarmIQ\device\services\vision-capture-2cam-service\calib\samples\left"
RIGHT_DIR = r"D:\FarmIQ\device\services\vision-capture-2cam-service\calib\samples\right"

aruco = cv2.aruco
dictionary = aruco.getPredefinedDictionary(aruco.DICT_6X6_250)

# สร้าง CharucoBoard (ต้องมีมาก่อน)
SQUARES_X = 10
SQUARES_Y = 7
SQUARE_LEN = 30.0
MARKER_LEN = 18.0
board = aruco.CharucoBoard((SQUARES_X, SQUARES_Y), SQUARE_LEN, MARKER_LEN, dictionary)
board_obj_points = board.getChessboardCorners()  # ใช้สำหรับ stereo

# detector
detector_params = aruco.DetectorParameters()
detector = aruco.ArucoDetector(dictionary, detector_params)

def list_images(folder):
    exts = ("*.png", "*.jpg", "*.jpeg", "*.bmp")
    files = []
    for e in exts:
        files.extend(glob.glob(os.path.join(folder, e)))
    return sorted(files)

left_images  = list_images(LEFT_DIR)
right_images = list_images(RIGHT_DIR)

assert len(left_images) == len(right_images), "จำนวนรูปซ้าย/ขวาไม่เท่ากัน"

print("จำนวนคู่ภาพ:", len(left_images))

all_charuco_corners_left  = []
all_charuco_ids_left      = []
all_charuco_corners_right = []
all_charuco_ids_right     = []

st_objpoints  = []  # 3D
st_imgpointsL = []  # 2D left
st_imgpointsR = []  # 2D right

image_size = None

for fL, fR in zip(left_images, right_images):
    imgL = cv2.imread(fL)
    imgR = cv2.imread(fR)

    if imgL is None or imgR is None:
        print(f"[WARN] อ่านรูปไม่ได้: {fL} หรือ {fR}")
        continue

    grayL = cv2.cvtColor(imgL, cv2.COLOR_BGR2GRAY)
    grayR = cv2.cvtColor(imgR, cv2.COLOR_BGR2GRAY)

    if image_size is None:
        image_size = grayL.shape[::-1]   # (w,h)

    # ---------- detect markers ฝั่งซ้าย ----------
    cornersL, idsL, _ = detector.detectMarkers(grayL)

    # ⭐ ถ้าไม่เจอ marker เลย ข้ามเฟรมนี้ไป
    if idsL is None or len(idsL) == 0:
        # print(f"[INFO] no markers in LEFT for {os.path.basename(fL)}")
        charucoCornersL, charucoIdsL = None, None
    else:
        retvalL, charucoCornersL, charucoIdsL = aruco.interpolateCornersCharuco(
            markerCorners=cornersL,
            markerIds=idsL,
            image=grayL,
            board=board
        )

    # ---------- detect markers ฝั่งขวา ----------
    cornersR, idsR, _ = detector.detectMarkers(grayR)

    if idsR is None or len(idsR) == 0:
        # print(f"[INFO] no markers in RIGHT for {os.path.basename(fR)}")
        charucoCornersR, charucoIdsR = None, None
    else:
        retvalR, charucoCornersR, charucoIdsR = aruco.interpolateCornersCharuco(
            markerCorners=cornersR,
            markerIds=idsR,
            image=grayR,
            board=board
        )

    # ---------- เก็บสำหรับ single-camera calibration ----------
    if charucoIdsL is not None and len(charucoIdsL) > 10:
        all_charuco_corners_left.append(charucoCornersL)
        all_charuco_ids_left.append(charucoIdsL)

    if charucoIdsR is not None and len(charucoIdsR) > 10:
        all_charuco_corners_right.append(charucoCornersR)
        all_charuco_ids_right.append(charucoIdsR)

    # ---------- เตรียมข้อมูลสำหรับ stereo ----------
    if (charucoIdsL is None or charucoIdsR is None):
        continue

    idsL_f = charucoIdsL.flatten()
    idsR_f = charucoIdsR.flatten()

    common_ids = np.intersect1d(idsL_f, idsR_f)
    if len(common_ids) < 10:
        continue

    objp  = []
    imgpL = []
    imgpR = []

    for cid in common_ids:
        idxL = np.where(idsL_f == cid)[0][0]
        idxR = np.where(idsR_f == cid)[0][0]

        # 3D point บนกระดาน
        objp.append(board_obj_points[cid])

        # 2D corner ในรูป
        imgpL.append(charucoCornersL[idxL][0])
        imgpR.append(charucoCornersR[idxR][0])

    objp  = np.array(objp,  dtype=np.float32)
    imgpL = np.array(imgpL, dtype=np.float32)
    imgpR = np.array(imgpR, dtype=np.float32)

    st_objpoints.append(objp)
    st_imgpointsL.append(imgpL)
    st_imgpointsR.append(imgpR)

print("เฟรมที่ใช้ calibrate กล้องซ้าย:", len(all_charuco_corners_left))
print("เฟรมที่ใช้ calibrate กล้องขวา:", len(all_charuco_corners_right))
print("เฟรมที่ใช้ stereo:", len(st_objpoints))


จำนวนคู่ภาพ: 102
เฟรมที่ใช้ calibrate กล้องซ้าย: 102
เฟรมที่ใช้ calibrate กล้องขวา: 102
เฟรมที่ใช้ stereo: 102


3.Calibrate intrinsic ของแต่ละกล้อง

In [27]:
# ====== Calibrate single camera (left) ======
print("\nCalibrating LEFT camera ...")
retL, K_L, dist_L, rvecs_L, tvecs_L, \
    stdInt_L, stdExt_L, perViewErr_L = aruco.calibrateCameraCharucoExtended(
        charucoCorners=all_charuco_corners_left,
        charucoIds=all_charuco_ids_left,
        board=board,
        imageSize=image_size,
        cameraMatrix=None,
        distCoeffs=None
)

print("RMS error LEFT:", retL)
print("K_L:\n", K_L)
print("dist_L:", dist_L.ravel())

# ====== Calibrate single camera (right) ======
print("\nCalibrating RIGHT camera ...")
retR, K_R, dist_R, rvecs_R, tvecs_R, \
    stdInt_R, stdExt_R, perViewErr_R = aruco.calibrateCameraCharucoExtended(
        charucoCorners=all_charuco_corners_right,
        charucoIds=all_charuco_ids_right,
        board=board,
        imageSize=image_size,
        cameraMatrix=None,
        distCoeffs=None
)

print("RMS error RIGHT:", retR)
print("K_R:\n", K_R)
print("dist_R:", dist_R.ravel())

fs = cv2.FileStorage("intrinsics_stereo.yml", cv2.FILE_STORAGE_WRITE)

fs.write("image_width",  int(image_size[0]))
fs.write("image_height", int(image_size[1]))

fs.write("K_left",  K_L)
fs.write("dist_left",  dist_L)
fs.write("K_right", K_R)
fs.write("dist_right", dist_R)

fs.release()
print("Saved intrinsics to intrinsics_stereo.yml")



Calibrating LEFT camera ...
RMS error LEFT: 0.5469551175463184
K_L:
 [[2.26075657e+03 0.00000000e+00 1.36942750e+03]
 [0.00000000e+00 2.26139055e+03 7.43031210e+02]
 [0.00000000e+00 0.00000000e+00 1.00000000e+00]]
dist_L: [-0.82951276  1.16058731  0.00176276 -0.00195205 -0.99540885]

Calibrating RIGHT camera ...
RMS error RIGHT: 0.5454110729370816
K_R:
 [[2.34792947e+03 0.00000000e+00 1.36059149e+03]
 [0.00000000e+00 2.35341395e+03 7.89767720e+02]
 [0.00000000e+00 0.00000000e+00 1.00000000e+00]]
dist_R: [-8.98195247e-01  1.39132315e+00 -5.10885168e-04 -1.48586865e-03
 -1.32138190e+00]
Saved intrinsics to intrinsics_stereo.yml


4.Stereo calibration หา R, T ระหว่างกล้อง

In [28]:
# ====== Stereo calibration ======
criteria = (cv2.TERM_CRITERIA_MAX_ITER + cv2.TERM_CRITERIA_EPS,
            100, 1e-5)

flags = cv2.CALIB_FIX_INTRINSIC   # ใช้ intrinsics ที่หาได้แล้ว

print("\nStereo calibrate ...")
retStereo, K_L2, dist_L2, K_R2, dist_R2, R, T, E, F = cv2.stereoCalibrate(
    objectPoints=st_objpoints,
    imagePoints1=st_imgpointsL,
    imagePoints2=st_imgpointsR,
    cameraMatrix1=K_L,
    distCoeffs1=dist_L,
    cameraMatrix2=K_R,
    distCoeffs2=dist_R,
    imageSize=image_size,
    criteria=criteria,
    flags=flags
)

print("RMS stereo:", retStereo)
print("R:\n", R)
print("T:\n", T)
print("Baseline length (mm):", np.linalg.norm(T))

# ====== Save intrinsics + stereo รวมไฟล์เดียว ======
calib_path = r"D:\FarmIQ\device\services\vision-capture-2cam-service\calib\intrinsics_stereo.yml"

fs = cv2.FileStorage(calib_path, cv2.FILE_STORAGE_WRITE)

# ขนาดภาพ
fs.write("image_width",  int(image_size[0]))
fs.write("image_height", int(image_size[1]))

# Intrinsics กล้องซ้าย–ขวา (ใช้ K_L / dist_L / K_R / dist_R ที่ได้จาก Charuco)
fs.write("K_left",      K_L)
fs.write("dist_left",   dist_L)
fs.write("K_right",     K_R)
fs.write("dist_right",  dist_R)

# Stereo extrinsics
fs.write("R", R)
fs.write("T", T)
fs.write("E", E)
fs.write("F", F)

fs.release()
print(f"Saved intrinsics + stereo params to: {calib_path}")



Stereo calibrate ...
RMS stereo: 2.1751683678246674
R:
 [[ 0.99950186 -0.03115563 -0.0050353 ]
 [ 0.03101508  0.99918266 -0.02592432]
 [ 0.00583888  0.02575523  0.99965123]]
T:
 [[1.14459536e+02]
 [3.97216282e-03]
 [2.33145694e+01]]
Baseline length (mm): 116.80990807516856
Saved intrinsics + stereo params to: D:\FarmIQ\device\services\vision-capture-2cam-service\calib\intrinsics_stereo.yml


5.เซฟผล calibration ไว้ใช้ตอนรันจริง

In [29]:
fs = cv2.FileStorage("stereo_charuco.yml", cv2.FILE_STORAGE_WRITE)
fs.write("image_width",  image_size[0])
fs.write("image_height", image_size[1])

fs.write("K_left",  K_L)
fs.write("dist_left",  dist_L)
fs.write("K_right", K_R)
fs.write("dist_right", dist_R)

fs.write("R", R)
fs.write("T", T)
fs.write("E", E)
fs.write("F", F)
fs.release()

print("Saved to stereo_charuco.yml")


Saved to stereo_charuco.yml


6.เตรียม rectification map ใช้ตอน capture จริง

In [33]:
print("\n=== Stereo Rectify & Rectify Maps ===")

alpha = 0.25  # 0 = crop เยอะ, 1 = เก็บ FOV เยอะ อาจมีขอบดำ

image_size = (2688, 1520)  # (width, height) จากภาพจริง

R1, R2, P1, P2, Q, roi1, roi2 = cv2.stereoRectify(
    K_L, dist_L, K_R, dist_R,
    image_size,
    R, T,
    flags=cv2.CALIB_ZERO_DISPARITY,
    alpha=alpha
)

mapLx, mapLy = cv2.initUndistortRectifyMap(
    K_L, dist_L, R1, P1, image_size, cv2.CV_32FC1
)
mapRx, mapRy = cv2.initUndistortRectifyMap(
    K_R, dist_R, R2, P2, image_size, cv2.CV_32FC1
)

print("mapLx shape:", mapLx.shape, "mapLy shape:", mapLy.shape)
print("mapRx shape:", mapRx.shape, "mapRy shape:", mapRy.shape)

fs = cv2.FileStorage("stereo_rectify_maps.yml", cv2.FILE_STORAGE_WRITE)
fs.write("mapLx", mapLx)
fs.write("mapLy", mapLy)
fs.write("mapRx", mapRx)
fs.write("mapRy", mapRy)
fs.write("Q", Q)
fs.release()
print("Saved to stereo_rectify_maps.yml")



=== Stereo Rectify & Rectify Maps ===
mapLx shape: (1520, 2688) mapLy shape: (1520, 2688)
mapRx shape: (1520, 2688) mapRy shape: (1520, 2688)
Saved to stereo_rectify_maps.yml
